# 0.8+ Audio Deepfake Detection Pipeline

GPU T4 연결 후 → **런타임 → 모두 실행** 하면 끝.

In [ ]:
# 셀 0: GPU 확인
!nvidia-smi

In [ ]:
# 셀 1: 환경 세팅
!pip -q install librosa soundfile transformers accelerate demucs panns-inference onnxruntime-gpu datasets huggingface_hub scikit-learn scipy edge-tts
!apt -qq install -y ffmpeg > /dev/null
print('Done')

In [ ]:
# 셀 2: GitHub에서 프로젝트 클론
!git clone https://github.com/jogwangjo/da1.git /content/da1
%cd /content/da1
!pwd
!ls -la

In [ ]:
# 셀 3: 필수 파일 확인
import os

required = [
    'scripts/build_train_data.py',
    'scripts/train_raptor.py',
    'submit/script_v2.py',
    'data/sample_submission.csv'
]

all_ok = True
for f in required:
    if os.path.exists(f):
        print(f'  OK: {f}')
    else:
        print(f'  MISSING: {f}')
        all_ok = False

if all_ok:
    print('\n✅ 모든 필수 파일 준비 완료!')
else:
    print('\n❌ 누락된 파일이 있습니다.')

In [ ]:
# 셀 4: 모델 파일 다운로드
import os
from huggingface_hub import hf_hub_download, snapshot_download

model_dir = 'submit/model'
os.makedirs(model_dir, exist_ok=True)

# DF-Arena 1B
df_dir = f'{model_dir}/df_arena_1b'
if not os.path.exists(f'{df_dir}/pytorch_model.bin'):
    print('Downloading DF-Arena 1B...')
    snapshot_download('shreshthgupta/DF-Arena-1B-Antispoofing', local_dir=df_dir)
    print('Done!')
else:
    print('DF-Arena 1B already exists')

# HTDemucs
htd_dir = f'{model_dir}/htdemucs'
if not os.path.exists(f'{htd_dir}/955717e8-8726e21a.th'):
    print('Downloading HTDemucs...')
    from pathlib import Path
    from demucs.pretrained import get_model
    m = get_model('htdemucs')
    os.makedirs(htd_dir, exist_ok=True)
    import torch
    torch.save(m.state_dict(), f'{htd_dir}/955717e8-8726e21a.th')
    # YAML config도 복사
    import shutil
    yaml_src = Path(m._config_path) if hasattr(m, '_config_path') else None
    if yaml_src and yaml_src.exists():
        shutil.copy(yaml_src, f'{htd_dir}/htdemucs.yaml')
    print('Done!')
else:
    print('HTDemucs already exists')

# SONICS
for name in ['sonics-alpha-5s', 'sonics-beta-5s']:
    sonics_dir = f'{model_dir}/{name}'
    if not os.path.exists(f'{sonics_dir}/pytorch_model.bin'):
        print(f'Downloading {name}...')
        snapshot_download('sagierte/sonics', allow_patterns=[f'{name}/*'], local_dir=model_dir)
        print('Done!')
    else:
        print(f'{name} already exists')

print('\n✅ 모델 다운로드 완료')

In [ ]:
# 셀 5: 학습 데이터 구축 (~30분)
# LibriSpeech 다운로드
import os
cache_dir = '_cache/librispeech'
if not os.path.exists(f'{cache_dir}/LibriSpeech/test-clean'):
    print('Downloading LibriSpeech test-clean...')
    !pip -q install audiocraft
    !python -c "
import os
from datasets import load_dataset
ds = load_dataset('librispeech_asr', 'clean', split='test', trust_remote_code=True)
out_dir = '{cache_dir}/LibriSpeech/test-clean'
os.makedirs(out_dir, exist_ok=True)
for i, sample in enumerate(ds):
    speaker = str(sample['speaker_id'])
    chapter = str(sample['chapter_id'])
    utt = str(sample['id']).split('_')[-1]
    subdir = os.path.join(out_dir, speaker, chapter)
    os.makedirs(subdir, exist_ok=True)
    import soundfile as sf
    import numpy as np
    audio = np.array(sample['audio']['array'], dtype=np.float32)
    sf.write(os.path.join(subdir, f'{speaker}-{chapter}-{utt}.flac'), audio, sample['audio']['sampling_rate'])
    if i >= 3000:
        break
print(f'Downloaded {i+1} files')
"
    print('LibriSpeech done!')
else:
    print('LibriSpeech already exists')

# 학습 데이터 구축
!python scripts/build_train_data.py \
    --out train_data \
    --auto-download \
    --max-voice-real 2000 \
    --max-voice-fake 3000 \
    --max-music-real 500 \
    --max-music-fake 2000

In [ ]:
# 셀 6: RAPTOR 학습 (~4시간)
!python scripts/train_raptor.py \
    --train train_data/manifest_train.csv \
    --val train_data/manifest_val.csv \
    --backbone utter-project/mHuBERT-147 \
    --out runs/raptor_v1 \
    --epochs 20 --bs 24 --lr 1e-6 \
    --lr-head 3e-4 --consistency-w 0.25 --p-aug 0.5

In [ ]:
# 셀 7: 학습 결과 확인
import pandas as pd
import os

log_path = 'runs/raptor_v1/log.csv'
if os.path.exists(log_path):
    log = pd.read_csv(log_path)
    print(log.to_string(index=False))
    print(f'\nBest val EER: {log["val_eer"].min():.4f}')
else:
    print(f'ERROR: {log_path} not found!')
    print('runs/raptor_v1/ 폴더 내용:')
    !ls -la runs/

In [ ]:
# 셀 8: 모델 복사 + 제출 추론
!cp runs/raptor_v1/best.pth submit/model/raptor_best.pth
!python submit/script_v2.py \
    --test-dir data/test \
    --sample-submission data/sample_submission.csv \
    --output output/submission.csv \
    --device cuda --tta 2

In [ ]:
# 셀 9: 제출 zip 생성
!python scripts/build_submit_zip.py \
    --script submit/script_v2.py \
    --output submit.zip

In [ ]:
# 셀 10: 결과 다운로드
from google.colab import files
files.download('submit.zip')
files.download('output/submission.csv')